In [1]:
cd ..

/Users/camila.cusicanqui/Documents/klar/mini-tasks


In [2]:
from data_ingest import get_db_conn
import pandas as pd
import numpy as np
import gspread
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict
import seaborn as sns
from datetime import date, timedelta, datetime
pd.set_option('display.max_columns', None)
from openai import OpenAI
# from import agents Agent, Runner          
import yaml
import os
import json 

In [3]:
import asyncio
from agents import Agent, Runner
from agents import Agent, Runner, function_tool
import datetime as dt
import polars as pl
import docx2txt
from pydantic import BaseModel, Field
from typing import List, Optional

In [4]:
# transform docx to text
process_context_txt = open("agents-automization/OPS-FRA-009 - Análisis de comercios fraudulentos.txt", "r+")
process_context_txt = process_context_txt.read()

In [5]:
# read yaml config file 


with open("/Users/camila.cusicanqui/Documents/klar/mini-tasks/.config/openai-credentials.yaml", "r") as file:
    config = yaml.safe_load(file)

api_key = config["key"]


In [6]:
os.environ["OPENAI_API_KEY"] = api_key

In [7]:
def common_commerces_from_cb(klrids: List[str], merchant_name = str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # query here that finds the transactions of the klrids and which commmerces they interacted with
    # klrids_revise = after_df["klrid"].unique()
    # merchant1_cards

    number_of_klrids = len(klrids)
    revise_trx_users_str = ",".join([f"'{user}'" for user in klrids])
    revise_trx_user_query = f"""
        with txn_base as (
            select
                t.user_id,
                t.timestamp_mx_created_at as trx_mx_time,
                t.source_external_id as klrid,
                t.type,
                t."state",
                t.merchant_name,
                t.description,
                t.transaction_id,
                t.amount,
                case
                    when upper(t.transaction_id) like 'PARABILIUM:%'
                        and nullif(split_part(t.transaction_id, ':', 2), '') ~ '^[0-9]+$'
                    then split_part(t.transaction_id, ':', 2)

                    when t.transaction_id ~ '^[0-9]+$'
                    then t.transaction_id

                    else null
                end as operation_id_str
            from analytics_bi.transactions t
            where
                t.source_external_id in ({revise_trx_users_str})
                and t.type = 'PURCHASE'
                and t.timestamp_mx_created_at >= '2026-02-10'::date
        ),

        txn_enriched as (
            select
                tb.*,
                o.omi_id_operacion,
                o.c022,
                o.c063,
                pt.t_c0,
                op.sucursal as merchant_affiliation,
                op.terminal as num_terminal,
                regexp_substr(o.c063, 'CE[0-9]5[^!]*') as ce_token
            from txn_base tb
            left join is_pii_parabilium.operation_iso_messages_temp o
                on o.omi_id_operacion = tb.operation_id_str
            left join
            is_pii_parabilium.operaciones op on op.id_operacion = tb.operation_id_str
            left join is_pii_parabilium.parabilium_transactions pt
                on pt.id = case
                    when tb.operation_id_str ~ '^[0-9]+$' then tb.operation_id_str::bigint
                    else null
                end
        ),

        txn_3ds as (
            select
                te.*,

                case
                    when substring(te.c022, 1, 2) = '00' then 'Unknown'
                    when substring(te.c022, 1, 2) = '01' then 'CNP Manual'
                    when substring(te.c022, 1, 2) = '05' then 'Integrated Circuit Read'
                    when substring(te.c022, 1, 2) = '07' then 'Contactless'
                    when substring(te.c022, 1, 2) = '10' then 'CNP Card On File'
                    when substring(te.c022, 1, 2) = '80' then 'Fallback'
                    when substring(te.c022, 1, 2) = '90' then 'Magnetic stripe read'
                    else null
                end as pos_entry_mode,

                case
                    when te.t_c0 is null then 'unknown'
                    when substring(te.t_c0, 32, 1) = '1' then 'has cvv'
                    else 'no cvv'
                end::varchar as cvv_ind,

                case
                    when te.ce_token is not null
                        and position('01' in te.ce_token) > 0
                    then substring(te.ce_token from position('01' in te.ce_token) + 2 for 2)
                    else null
                end as leading_indicator
            from txn_enriched te
        )

        select
            tc.*,
            case
                when tc.ce_token is null then 'NO_3DS'
                when tc.leading_indicator in ('kA','kB','kC','kE','kF','kJ','kR','kS','kG','kO','kP')
                    then '3DS_AUTHENTICATED'
                when tc.leading_indicator in ('kN','kW','kU','kX')
                    then '3DS_NOT_AUTHENTICATED'
                else 'UNKNOWN'
            end as three_ds_status,

            case
                when tc.ce_token is null then 'NO_3DS'
                when tc.leading_indicator in ('kA','kC','kE','kO')
                    then 'FRICTIONLESS'
                when tc.leading_indicator in ('kB','kS','kG','kP')
                    then 'CHALLENGE'
                when tc.leading_indicator in ('kN','kW','kU','kX')
                    then 'EXEMPT_OR_INFO'
                else 'UNKNOWN'
            end as three_ds_flow,

            c.product_type,
            c.card_type,
            c.status
        from txn_3ds tc
        left join is_provider_parabilium.cards c
            on c.id = tc.klrid

    """
    revise_users_df = pd.read_sql(revise_trx_user_query, get_db_conn())
    revise_users_df["country"] = revise_users_df["merchant_name"].str[-2:]
    revise_users_df.sort_values(by=['klrid','trx_mx_time'], inplace=True)
    top_commerces = revise_users_df.groupby(["merchant_name"])["klrid"].nunique().sort_values(ascending=False).reset_index(name="unique_klrids")
    top_commerces = top_commerces[(top_commerces["merchant_name"] != merchant_name) & (top_commerces["unique_klrids"] > np.floor(number_of_klrids*0.75))]
    revise_users_df = revise_users_df[(revise_users_df["merchant_name"] == merchant_name) | revise_users_df["merchant_name"].isin(top_commerces["merchant_name"].unique())]
    return top_commerces, revise_users_df

In [8]:
# def chargeback_merchant_summary(merchants_to_block: List[str]) -> Dict[str, Dict]:


#     def _series_to_dict(series: pd.Series) -> Dict:
#         return {str(idx): int(value) for idx, value in series.fillna("null").items()}

#     def _dataframe_to_records(df: pd.DataFrame) -> List[Dict]:
#         if df.empty:
#             return []

#         serializable_df = df.copy()
#         for column in serializable_df.columns:
#             if pd.api.types.is_datetime64_any_dtype(serializable_df[column]):
#                 serializable_df[column] = serializable_df[column].astype(str)

#         return serializable_df.to_dict(orient="records")

#     report = {}
#     timeframes = {
#         "1_week": (dt.datetime.now() - dt.timedelta(days=7)).strftime('%Y-%m-%d %H:%M:%S'),
#         "1_month": (dt.datetime.now() - dt.timedelta(days=30)).strftime('%Y-%m-%d %H:%M:%S'),
#         "ytd": dt.datetime(dt.datetime.now().year, 1, 1).strftime('%Y-%m-%d %H:%M:%S'),
#     }

#     for merchant in merchants_to_block:
#         merchant_name = merchant.replace('*', '')
#         merchant_cb_df = cb_df[cb_df["merchant"] == merchant_name].copy()

#         if merchant_cb_df.empty:
#             report[merchant_name] = {
#                 "merchant_name": merchant_name,
#                 "first_appearance": None,
#                 "last_appearance": None,
#                 "shared_commerces_summary": [],
#                 "shared_commerces_transactions": [],
#                 "timeframes": {},
#             }
#             continue

#         klrids_merchant = merchant_cb_df["klrid"].unique().tolist()
#         top_commerces, revise_users_df = common_commerces_from_cb(
#             klrids=klrids_merchant,
#             merchant_name=merchant_name,
#         )

#         merchant_summary = {
#             "merchant_name": merchant_name,
#             "first_appearance": str(merchant_cb_df["trx_timestamp_mx"].min()),
#             "last_appearance": str(merchant_cb_df["trx_timestamp_mx"].max()),
#             "shared_commerces_summary": _dataframe_to_records(top_commerces),
#             "shared_commerces_transactions": _dataframe_to_records(revise_users_df),
#             "timeframes": {},
#         }

#         for timeframe_name, start_time in timeframes.items():
#             timeframe_df = merchant_cb_df[merchant_cb_df["trx_timestamp_mx"] >= start_time]
#             merchant_summary["timeframes"][timeframe_name] = {
#                 "start_time": start_time,
#                 "num_chargebacks_reported": int(timeframe_df["transaction_id"].nunique()),
#                 "num_unique_users": int(timeframe_df["user_id"].nunique()),
#                 "num_klrids_per_card_type": _series_to_dict(timeframe_df.groupby("card_type")["klrid"].nunique()),
#                 "num_klrids_per_product_type": _series_to_dict(timeframe_df.groupby("product_type")["klrid"].nunique()),
#                 "num_transactions_per_cvv_indicator": _series_to_dict(timeframe_df.groupby("cvv_ind")["transaction_id"].nunique()),
#                 "num_transactions_per_3ds_status": _series_to_dict(timeframe_df.groupby("three_ds_status")["transaction_id"].nunique()),
#             }

#         report[merchant_name] = merchant_summary

#     return report

In [9]:
class FraudFinding(BaseModel):
    merchant_name: str
    risk_level: str = Field(description="low, medium, high")
    summary: str
    suspicious_signals: List[str]
    recommendation: str
    confidence: float

## Agent

In [10]:
# add requirements of blocking
# safety net alerts and ADC alerts of copying information

In [11]:
# add definition of response codes
# conectarme al google sheets BL Merchants
# hay otro comercio involucrado del comercio ?
    # los otros comercios en común están bloqueados o no ?
# deadline
    # what we need
    # what we're going to work on 
# expected deadline for the project

# Full implementation AI + text summaries + testing

*Logic*

1. receive the merchant

2. run the necessary prompts and sections

3. construct the final prompt

4. build the for statement to receive the merchant name in the list


### example merchants


# Full pipeline

In [ ]:
def build_merchant_activity_query(merchant_name: str, blocked_at_mx: str) -> str:
    """
    Builds the query used to get activity for a blocked merchant up to the current
    blocked timestamp in Mexico City timezone.
    """
    merchant_sql = merchant_name.replace("'", "''")
    # convert blocked date str to datetime
    blocked_at_mx = dt.datetime.strptime(blocked_at_mx, "%Y-%m-%d %H:%M:%S")
    blocked_at_sql = blocked_at_mx.strftime('%Y-%m-%d %H:%M:%S')

    return f"""
        WITH ytd_base AS (
        SELECT DISTINCT
            COALESCE(NULLIF(SPLIT_PART(transaction_id, 'PARABILIUM:', 2), ''), transaction_id) AS transaction_id_clean,
            user_id,
            state
        FROM analytics_bi.transactions__ytd
        WHERE type = 'PURCHASE'
        )
        SELECT
        CAST(ot.id_operacion AS VARCHAR(100)) AS transaction_id,
        convert_timezone('UTC','America/Mexico_City', ot.fecharegistro) AS tx_date_mx,
        CAST(convert_timezone('UTC','America/Mexico_City', ot.fecharegistro) AS DATE) AS tx_date_mx_date,
        ot.operador AS merchant_name,
        CAST(TRIM(ot.cod_respuesta) AS INTEGER) AS cod_respuesta,
        y.user_id,
        y.state,
        c.card_type,
        c.product_type,
        CASE
            WHEN pt.t_c0 IS NULL THEN 'unknown'
            WHEN SUBSTRING(pt.t_c0, 32, 1) = '1' THEN 'has cvv'
            ELSE 'no cvv'
        END AS cvv,
        CASE
            WHEN POSITION('! C0' IN oimt.c063) > 0
            THEN SUBSTRING(oimt.c063, POSITION('! C0' IN oimt.c063) + 28, 1)
            ELSE NULL
        END AS eci
        FROM is_pii_parabilium.operations_temp ot
        LEFT JOIN is_pii_parabilium.operaciones o
        ON o.id_operacion = ot.id_operacion
        LEFT JOIN is_provider_parabilium.cards c
        ON c.id = o.klrid
        LEFT JOIN is_pii_parabilium.operation_iso_messages_temp oimt
        ON oimt.omi_id_operacion = ot.id_operacion
        LEFT JOIN is_pii_parabilium.parabilium_transactions pt
        ON pt.id = ot.id_operacion
        LEFT JOIN ytd_base y
        ON y.transaction_id_clean = CAST(ot.id_operacion AS VARCHAR(100))
        WHERE ot.operador = '{merchant_sql}'
        AND convert_timezone('UTC','America/Mexico_City', ot.fecharegistro) <= TIMESTAMP '{blocked_at_sql}'
        AND EXTRACT(YEAR FROM convert_timezone('UTC','America/Mexico_City', ot.fecharegistro)) = EXTRACT(YEAR FROM TIMESTAMP '{blocked_at_sql}');
            """

In [13]:
def build_merchant_activity_summary(raw_df: pl.DataFrame, blocked_at_mx: str) -> pl.DataFrame:
    """
    Builds the merchant activity summary in Polars for T1D, T1W and YTD windows.
    """
    rows = []
    target_response_codes = [0, 14, 87, 51]
    blocked_at_dt = dt.datetime.strptime(blocked_at_mx, "%Y-%m-%d %H:%M:%S")
    block_date_mx = blocked_at_dt.date()
    week_start_mx = block_date_mx - dt.timedelta(days=6)
    year_start_mx = dt.date(block_date_mx.year, 1, 1)

    def safe_value(value):
        if value is None:
            return None
        if isinstance(value, dt.datetime):
            return value.strftime('%Y-%m-%d %H:%M')
        return value

    def should_keep_metric(t1d, t1w, ytd) -> bool:
        values = [t1d, t1w, ytd]
        return any(value not in [None, 0, '0'] for value in values)

    def append_metric(metric_name: str, t1d, t1w, ytd):
        if should_keep_metric(t1d, t1w, ytd):
            rows.append({
                'metric': metric_name,
                't1d': safe_value(t1d),
                't1w': safe_value(t1w),
                'ytd': safe_value(ytd)
            })

    if raw_df.is_empty():
        return pl.DataFrame({
            'metric': [],
            't1d': [],
            't1w': [],
            'ytd': []
        })

    df = raw_df.with_columns([
        pl.col('tx_date_mx').cast(pl.Datetime, strict=False).alias('tx_date_mx_parsed'),
        pl.col('tx_date_mx_date').cast(pl.Date, strict=False).alias('tx_date_mx_date_parsed'),
        pl.col('cod_respuesta').cast(pl.Int64, strict=False).alias('cod_respuesta_int'),
        pl.col('user_id').cast(pl.Utf8, strict=False).alias('user_id_cast'),
        pl.col('state').cast(pl.Utf8, strict=False).alias('state_cast'),
        pl.col('card_type').cast(pl.Utf8, strict=False).alias('card_type_cast'),
        pl.col('product_type').cast(pl.Utf8, strict=False).alias('product_type_cast'),
        pl.col('cvv').cast(pl.Utf8, strict=False).alias('cvv_cast'),
        pl.col('eci').cast(pl.Utf8, strict=False).alias('eci_cast')
    ]).with_columns([
        (pl.col('tx_date_mx_date_parsed') == pl.lit(block_date_mx)).alias('flg_t1d'),
        (
            (pl.col('tx_date_mx_date_parsed') >= pl.lit(week_start_mx)) &
            (pl.col('tx_date_mx_date_parsed') <= pl.lit(block_date_mx))
        ).alias('flg_t1w'),
        (
            (pl.col('tx_date_mx_date_parsed') >= pl.lit(year_start_mx)) &
            (pl.col('tx_date_mx_date_parsed') <= pl.lit(block_date_mx))
        ).alias('flg_ytd'),
        (
            (pl.col('cvv_cast') == 'no cvv') &
            (~pl.col('eci_cast').fill_null('').is_in(['5', '6']))
        ).alias('flg_no_3ds_no_cvv'),
        (
            (pl.col('cvv_cast') == 'has cvv') |
            (pl.col('eci_cast').fill_null('').is_in(['5', '6']))
        ).alias('flg_has_cvv_or_3ds')
    ])

    def count_for(mask_expr, period_expr):
        return df.filter(mask_expr & period_expr).height

    def first_for(period_expr):
        return df.filter(period_expr).select(pl.col('tx_date_mx_parsed').min()).item()

    def last_for(period_expr):
        return df.filter(period_expr).select(pl.col('tx_date_mx_parsed').max()).item()

    def distinct_users_for(period_expr):
        value = (
            df.filter(period_expr & pl.col('user_id_cast').is_not_null())
                .select(pl.col('user_id_cast').n_unique())
                .item()
        )
        return 0 if value is None else int(value)

    append_metric(
        'first_tx_date',
        first_for(pl.col('flg_t1d')),
        first_for(pl.col('flg_t1w')),
        first_for(pl.col('flg_ytd'))
    )
    append_metric(
        'last_tx_date',
        last_for(pl.col('flg_t1d')),
        last_for(pl.col('flg_t1w')),
        last_for(pl.col('flg_ytd'))
    )

    for code in target_response_codes:
        code_mask = pl.col('cod_respuesta_int') == code
        append_metric(
            f'cnt_cr_{code}',
            count_for(code_mask, pl.col('flg_t1d')),
            count_for(code_mask, pl.col('flg_t1w')),
            count_for(code_mask, pl.col('flg_ytd'))
        )
        append_metric(
            f'cnt_cr_{code}_no_3ds_no_cvv',
            count_for(code_mask & pl.col('flg_no_3ds_no_cvv'), pl.col('flg_t1d')),
            count_for(code_mask & pl.col('flg_no_3ds_no_cvv'), pl.col('flg_t1w')),
            count_for(code_mask & pl.col('flg_no_3ds_no_cvv'), pl.col('flg_ytd'))
        )
        append_metric(
            f'cnt_cr_{code}_has_cvv_or_has_3ds',
            count_for(code_mask & pl.col('flg_has_cvv_or_3ds'), pl.col('flg_t1d')),
            count_for(code_mask & pl.col('flg_has_cvv_or_3ds'), pl.col('flg_t1w')),
            count_for(code_mask & pl.col('flg_has_cvv_or_3ds'), pl.col('flg_ytd'))
        )

    append_metric(
        'cnt_unique_users',
        distinct_users_for(pl.col('flg_t1d')),
        distinct_users_for(pl.col('flg_t1w')),
        distinct_users_for(pl.col('flg_ytd'))
    )
    append_metric(
        'cnt_state_failed_canceled',
        count_for(pl.col('state_cast').is_in(['FAILED', 'CANCELED']), pl.col('flg_t1d')),
        count_for(pl.col('state_cast').is_in(['FAILED', 'CANCELED']), pl.col('flg_t1w')),
        count_for(pl.col('state_cast').is_in(['FAILED', 'CANCELED']), pl.col('flg_ytd'))
    )
    append_metric(
        'cnt_authorized_settled',
        count_for(pl.col('state_cast').is_in(['AUTHORIZED', 'SETTLED']), pl.col('flg_t1d')),
        count_for(pl.col('state_cast').is_in(['AUTHORIZED', 'SETTLED']), pl.col('flg_t1w')),
        count_for(pl.col('state_cast').is_in(['AUTHORIZED', 'SETTLED']), pl.col('flg_ytd'))
    )

    card_types = (
        df.filter(
            pl.col('card_type_cast').is_not_null() &
            (pl.col('card_type_cast').map_elements(lambda x: x.strip() if x is not None else '', return_dtype=pl.Utf8) != '')
        )
        .select('card_type_cast')
        .unique()
        .sort('card_type_cast')
        .to_series()
        .to_list()
    )

    for card_type in card_types:
        card_mask = (
            (pl.col('card_type_cast') == card_type) &
            pl.col('flg_no_3ds_no_cvv')
        )
        append_metric(
            f'cnt_card_type_{card_type}_no_3ds_no_cvv',
            count_for(card_mask, pl.col('flg_t1d')),
            count_for(card_mask, pl.col('flg_t1w')),
            count_for(card_mask, pl.col('flg_ytd'))
        )

    product_types = (
        df.filter(
            pl.col('product_type_cast').is_not_null() &
            (pl.col('product_type_cast').map_elements(lambda x: x.strip() if x is not None else '', return_dtype=pl.Utf8) != '')
        )
        .select('product_type_cast')
        .unique()
        .sort('product_type_cast')
        .to_series()
        .to_list()
    )

    for product_type in product_types:
        product_mask = (
            (pl.col('product_type_cast') == product_type) &
            pl.col('flg_no_3ds_no_cvv')
        )
        append_metric(
            f'cnt_product_type_{product_type}_no_3ds_no_cvv',
            count_for(product_mask, pl.col('flg_t1d')),
            count_for(product_mask, pl.col('flg_t1w')),
            count_for(product_mask, pl.col('flg_ytd'))
        )

    return pl.DataFrame(rows) if rows else pl.DataFrame({
        'metric': [],
        't1d': [],
        't1w': [],
        'ytd': []
    })

In [14]:
def merchant_trx_summary(merchant:str, blocked_date:str) -> pl.DataFrame:
    merchant_activity_query = build_merchant_activity_query(merchant, blocked_date)
    merchant_activity_raw = pl.read_database(
        query=merchant_activity_query,
        connection=get_db_conn()
    )
    return build_merchant_activity_summary(merchant_activity_raw, blocked_date)

In [15]:
# def find_chargeback(merchant_to_block: str) -> pd.DataFrame:
#     # run query here
#     cb_query = f"""
#         WITH tc_base AS (
#             SELECT
#                 tc.*,
#                 COALESCE(
#                     NULLIF(split_part(tc.transaction_id, 'PARABILIUM:', 2), ''),
#                     tc.transaction_id
#                 ) AS omi_id_operacion_raw
#             FROM ops_fraud.total_chargeback tc
#             --where
#             --    tc.merchant ilike '%' || '{merchant_to_block}' || '%'
#         ),

#         oimt_base AS (
#             SELECT
#                 oimt.omi_id_operacion,
#                 oimt.c063,
#                 oimt.c032 as adquirente
#             FROM is_pii_parabilium.operation_iso_messages_temp oimt
#             INNER JOIN tc_base tc
#                 ON oimt.omi_id_operacion = tc.omi_id_operacion_raw
#         ),

#         pin_verif AS (
#             SELECT
#                 o.omi_id_operacion,
#                 /* ---------- CVM RESULTS (Byte 1) ---------- */
#                 '! ' || REGEXP_SUBSTR(o.c063, 'B300080[^!]*') AS B300080_value,
#                 SUBSTRING(B300080_value FROM 49 FOR 6) AS "8-CVMRSLTS",
#                 SUBSTRING("8-CVMRSLTS" FROM 1 FOR 2) AS byte_1_hex,
#                 CASE
#                     WHEN byte_1_hex ~ '^[0-9A-Fa-f]{2}$'
#                     THEN FROM_VARBYTE(from_hex(byte_1_hex), 'binary')
#                     ELSE NULL
#                 END AS byte_1_bits,
#                 CASE
#                     WHEN byte_1_bits IS NULL THEN NULL
#                     WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000000'
#                         THEN 'Procesamiento de CVM fallido'
#                     WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000001'
#                         THEN 'Verificacion de PIN en texto plano realizada por el chip de la tarjeta'
#                     WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000010'
#                         THEN 'PIN cifrado verificado en linea'
#                     WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000011'
#                         THEN 'Verificacion de PIN en texto plano realizada por el chip de la tarjeta y firma (papel)'
#                     WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000100'
#                         THEN 'Verificacion de PIN cifrado realizada por el chip de la tarjeta'
#                     WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000101'
#                         THEN 'Verificacion de PIN cifrado realizada por el chip de la tarjeta y firma (papel)'
#                     WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '011110'
#                         THEN 'Firma (papel)'
#                     WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '011111'
#                         THEN 'No se requiere CVM'
#                     ELSE SUBSTRING(byte_1_bits FROM 3 FOR 6)
#                 END AS metodo_verificacion
#             FROM oimt_base o
#             WHERE o.c063 LIKE '%! B3%'
#         ),

#         three_ds AS (
#             SELECT
#                 o.omi_id_operacion,
#                 o.c063,

#                 /* ---------- CE TOKEN EXTRACTION ---------- */
#                 REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*') AS ce_token,

#                 /* ---------- LEADING INDICATOR (kA, kG, kN, etc.) ---------- */
#                 CASE
#                     WHEN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*') IS NOT NULL
#                         AND position('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) > 0
#                     THEN substring(
#                         REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
#                         FROM position('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
#                         FOR 2
#                     )
#                     ELSE NULL
#                 END AS leading_indicator,

#                 /* ---------- 3DS STATUS ---------- */
#                 CASE
#                     WHEN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*') IS NULL
#                         THEN 'NO_3DS'
#                     WHEN substring(
#                             REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
#                             FROM position('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
#                             FOR 2
#                         ) IN ('kA','kB','kC','kE','kF','kJ','kR','kS','kG','kO','kP')
#                         THEN '3DS_AUTHENTICATED'
#                     WHEN substring(
#                             REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
#                             FROM position('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
#                             FOR 2
#                         ) IN ('kN','kW','kU','kX')
#                         THEN '3DS_NOT_AUTHENTICATED'
#                     ELSE 'UNKNOWN'
#                 END AS three_ds_status,

#                 /* ---------- 3DS FLOW ---------- */
#                 CASE
#                     WHEN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*') IS NULL
#                         THEN 'NO_3DS'
#                     WHEN substring(
#                             REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
#                             FROM position('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
#                             FOR 2
#                         ) IN ('kA','kC','kE','kO')
#                         THEN 'FRICTIONLESS'
#                     WHEN substring(
#                             REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
#                             FROM position('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
#                             FOR 2
#                         ) IN ('kB','kS','kG','kP')
#                         THEN 'CHALLENGE'
#                     WHEN substring(
#                             REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
#                             FROM position('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
#                             FOR 2
#                         ) IN ('kN','kW','kU','kX')
#                         THEN 'EXEMPT_OR_INFO'
#                     ELSE 'UNKNOWN'
#                 END AS three_ds_flow
#             FROM oimt_base o
#         )

#         SELECT
#             tc.*,
#             pv.metodo_verificacion AS metodo_identificacion,
#             td.leading_indicator,
#             td.three_ds_status,
#             td.three_ds_flow,
#             td.c063,
#             o.sucursal as afiliacion,
#             o.terminal as numero_terminal,
#             ob.adquirente
#         FROM tc_base tc
#         LEFT JOIN pin_verif pv
#             ON pv.omi_id_operacion = tc.omi_id_operacion_raw
#         LEFT JOIN three_ds td
#             ON td.omi_id_operacion = tc.omi_id_operacion_raw
#         left join
#             is_pii_parabilium.parabilium_transactions pt on pt.id = tc.omi_id_operacion_raw
#         left join
#             is_pii_parabilium.operaciones o on o.id_operacion = tc.omi_id_operacion_raw
#         LEFT JOIN oimt_base ob
#             ON ob.omi_id_operacion = tc.omi_id_operacion_raw
#         """

#     cb_df = pd.read_sql(cb_query, get_db_conn())
#     cb_df = cb_df[cb_df["merchant"] == merchant_to_block]
#     if cb_df.empty:
#         return cb_df
#     else:
#         cb_df["trx_time"] = cb_df['trx_timestamp_mx']
#         cb_df['trx_timestamp_mx'] = pd.to_datetime(cb_df['trx_timestamp_mx'])
#         cb_df['week_number'] = cb_df['trx_timestamp_mx'].dt.isocalendar().week
#         cb_df['trx_date'] = cb_df["trx_timestamp_mx"].dt.date
#         cb_df['trx_month'] = cb_df['trx_timestamp_mx'].dt.month
#         cb_df['trx_day'] = cb_df["trx_timestamp_mx"].dt.day
#         cb_df['trx_month_year'] = cb_df['trx_timestamp_mx'].dt.to_period('M')
#         cb_df['trx_week_year'] = cb_df['trx_timestamp_mx'].dt.strftime('%Y-%W')
#         # cb timestamp
#         cb_df['cb_date'] = cb_df["cb_timestamp"].dt.date
#         cb_df['cb_month'] = cb_df['cb_timestamp'].dt.month
#         cb_df['cb_day'] = cb_df["cb_timestamp"].dt.day
#         cb_df['cb_month_year'] = cb_df['cb_timestamp'].dt.to_period('M')
#         cb_df['cb_week_year'] = cb_df['cb_timestamp'].dt.strftime('%Y-%W')
#         cb_df.drop_duplicates(subset=["transaction_id"], inplace=True, keep='first')
#         cb_df = cb_df[(cb_df["transaction_id"].notna()) &(cb_df["amount"].notna())]
#         # mapear cb_reasons to the correct ones
#         cb_df = cb_df[cb_df["cb_reason"] != 'Devolución de dinero']
#         cb_df["cb_reason_original"] = cb_df["cb_reason"]
#         cb_df["cb_reason"] = cb_df["cb_reason"].map({
#             'Transacción por internet o compra telefónica': "Rembolso no recibido",
#             'Error durante el proceso': "Cargo no reconocido",
#             'Otro' : "Otro",
#             'Producto no recibido': "Rembolso no recibido",
#             'Transacción duplicada': 'Transacción duplicada',
#             'Transacción declinada': 'Transacción declinada',
#             'Transacción excede el importe autorizado': 'Rembolso no procesado',
#             'Aclaración de Transacción de Cajero Automático': 'Aclaración de Transacción de Cajero Automático',
#             'ATM no entrega dinero': 'Aclaración de Transacción de Cajero Automático',
#             'Reporte de cargo no reconocido':  "Cargo no reconocido",
#             'TRANSACTION_EXCEEDS_AUTHORIZED_AMOUNT':  'Rembolso no procesado',
#             'Reembolso no procesado': 'Rembolso no procesado',
#             'REFUND_NOT_PROCESSED': 'Rembolso no procesado',
#             'TRANSACTION_DECLINED': 'Transacción declinada',
#             'OTHER': 'Otro',
#             'UNRECOGNIZED_TRANSACTION': 'Cargo no reconocido',
#             'INTERNET_PHONE_TRANSACTION': 'Rembolso no recibido',
#             'UNRECOGNIZED_CHARGE': "Cargo no reconocido",
#             'DUPLICATE_TRANSACTION': 'Transacción duplicada',
#             'ATM_TRANSACTION_CLARIFICATION': 'Aclaración de Transacción de Cajero Automático',
#         })
#         # obtain last two characters of merchant
#         cb_df["country"] = cb_df.merchant.str[-2:]
#         return cb_df

In [16]:
# def chargeback_merchant_summary(merchant_name: str) -> Dict[str, Dict]:
#     def _series_to_dict(series: pd.Series) -> Dict:
#         return {str(idx): int(value) for idx, value in series.fillna("null").items()}

#     def _dataframe_to_records(df: pd.DataFrame) -> List[Dict]:
#         if df.empty:
#             return []

#         serializable_df = df.copy()
#         for column in serializable_df.columns:
#             if pd.api.types.is_datetime64_any_dtype(serializable_df[column]):
#                 serializable_df[column] = serializable_df[column].astype(str)

#         return serializable_df.to_dict(orient="records")

#     report = {}
#     timeframes = {
#         "1_week": (dt.datetime.now() - dt.timedelta(days=7)).strftime('%Y-%m-%d %H:%M:%S'),
#         "1_month": (dt.datetime.now() - dt.timedelta(days=30)).strftime('%Y-%m-%d %H:%M:%S'),
#         "ytd": dt.datetime(dt.datetime.now().year, 1, 1).strftime('%Y-%m-%d %H:%M:%S'),
#     }
#     merchant_name = merchant_name.replace('*', '')
#     merchant_cb_df = find_chargeback(merchant_name)

#     if merchant_cb_df.empty:
#         report[merchant_name] = {
#             "merchant_name": merchant_name,
#             "first_appearance": None,
#             "last_appearance": None,
#             "shared_commerces_summary": [],
#             "shared_commerces_transactions": [],
#             "timeframes": {},
#         }
#         return report
#     else:
#         klrids_merchant = merchant_cb_df["klrid"].unique().tolist()
#         # top_commerces, revise_users_df = common_commerces_from_cb(
#         #     klrids=klrids_merchant,
#         #     merchant_name=merchant_name,
#         # )

#         merchant_summary = {
#             "merchant_name": merchant_name,
#             "first_appearance": str(merchant_cb_df["trx_timestamp_mx"].min()),
#             "last_appearance": str(merchant_cb_df["trx_timestamp_mx"].max()),
#             # "shared_commerces_summary": _dataframe_to_records(top_commerces),
#             # "shared_commerces_transactions": _dataframe_to_records(revise_users_df),
#             "timeframes": {},
#         }

#         for timeframe_name, start_time in timeframes.items():
#             timeframe_df = merchant_cb_df[merchant_cb_df["trx_timestamp_mx"] >= start_time]
#             merchant_summary["timeframes"][timeframe_name] = {
#                 "start_time": start_time,
#                 "num_chargebacks_reported": int(timeframe_df["transaction_id"].nunique()),
#                 "num_unique_users": int(timeframe_df["user_id"].nunique()),
#                 "num_klrids_per_card_type": _series_to_dict(timeframe_df.groupby("card_type")["klrid"].nunique()),
#                 "num_klrids_per_product_type": _series_to_dict(timeframe_df.groupby("product_type")["klrid"].nunique()),
#                 "num_transactions_per_cvv_indicator": _series_to_dict(timeframe_df.groupby("cvv_ind")["transaction_id"].nunique()),
#                 "num_transactions_per_3ds_status": _series_to_dict(timeframe_df.groupby("three_ds_status")["transaction_id"].nunique()),
#             }

#         report[merchant_name] = merchant_summary

#     return report

In [17]:
def _series_to_dict(series: pd.Series) -> Dict[str, int]:
    return {str(idx): int(value) for idx, value in series.fillna("null").items()}


def _normalize_blocking_timestamp(
    blocking_date: str | dt.date | dt.datetime | pd.Timestamp,
) -> pd.Timestamp:
    blocking_ts = pd.Timestamp(blocking_date)

    if isinstance(blocking_date, dt.date) and not isinstance(blocking_date, dt.datetime):
        return blocking_ts.normalize() + pd.Timedelta(days=1) - pd.Timedelta(microseconds=1)

    blocking_text = str(blocking_date).strip()
    if len(blocking_text) <= 10:
        return blocking_ts.normalize() + pd.Timedelta(days=1) - pd.Timedelta(microseconds=1)

    return blocking_ts


def _normalize_blocking_day(
    blocking_date: str | dt.date | dt.datetime | pd.Timestamp,
) -> dt.date:
    return pd.Timestamp(blocking_date).date()


def _build_timeframes(
    blocking_date: str | dt.date | dt.datetime | pd.Timestamp,
) -> Dict[str, Dict[str, pd.Timestamp]]:
    blocking_ts = _normalize_blocking_timestamp(blocking_date)

    return {
        "1_week": {
            "start_time": blocking_ts - pd.Timedelta(days=7),
            "end_time": blocking_ts,
        },
        "1_month": {
            "start_time": blocking_ts - pd.Timedelta(days=30),
            "end_time": blocking_ts,
        },
        "ytd": {
            "start_time": pd.Timestamp(dt.datetime(blocking_ts.year, 1, 1)),
            "end_time": blocking_ts,
        },
    }


def find_chargeback(
    merchant_to_block: str,
    blocking_date: str | dt.date | dt.datetime | pd.Timestamp,
) -> pd.DataFrame:
    merchant_name = merchant_to_block.replace("*", "")
    blocking_day_sql = _normalize_blocking_day(blocking_date).isoformat()

    cb_query = f"""
        WITH tc_base AS (
            SELECT
                tc.*,
                COALESCE(
                    NULLIF(split_part(tc.transaction_id, 'PARABILIUM:', 2), ''),
                    tc.transaction_id
                ) AS omi_id_operacion_raw
            FROM ops_fraud.total_chargeback tc
            WHERE
                tc.merchant = '{merchant_name}'
                AND tc.trx_timestamp_mx < DATEADD(day, 1, DATE '{blocking_day_sql}')
        ),

        oimt_base AS (
            SELECT
                oimt.omi_id_operacion,
                oimt.c063,
                oimt.c032 AS adquirente
            FROM is_pii_parabilium.operation_iso_messages_temp oimt
            INNER JOIN tc_base tc
                ON oimt.omi_id_operacion = tc.omi_id_operacion_raw
        ),

        pin_verif AS (
            SELECT
                o.omi_id_operacion,
                '! ' || REGEXP_SUBSTR(o.c063, 'B300080[^!]*') AS B300080_value,
                SUBSTRING(B300080_value FROM 49 FOR 6) AS "8-CVMRSLTS",
                SUBSTRING("8-CVMRSLTS" FROM 1 FOR 2) AS byte_1_hex,
                CASE
                    WHEN byte_1_hex ~ '^[0-9A-Fa-f]{2}$'
                    THEN FROM_VARBYTE(from_hex(byte_1_hex), 'binary')
                    ELSE NULL
                END AS byte_1_bits,
                CASE
                    WHEN byte_1_bits IS NULL THEN NULL
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000000'
                        THEN 'Procesamiento de CVM fallido'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000001'
                        THEN 'Verificacion de PIN en texto plano realizada por el chip de la tarjeta'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000010'
                        THEN 'PIN cifrado verificado en linea'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000011'
                        THEN 'Verificacion de PIN en texto plano realizada por el chip de la tarjeta y firma (papel)'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000100'
                        THEN 'Verificacion de PIN cifrado realizada por el chip de la tarjeta'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '000101'
                        THEN 'Verificacion de PIN cifrado realizada por el chip de la tarjeta y firma (papel)'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '011110'
                        THEN 'Firma (papel)'
                    WHEN SUBSTRING(byte_1_bits FROM 3 FOR 6) = '011111'
                        THEN 'No se requiere CVM'
                    ELSE SUBSTRING(byte_1_bits FROM 3 FOR 6)
                END AS metodo_verificacion
            FROM oimt_base o
            WHERE o.c063 LIKE '%! B3%'
        ),

        three_ds AS (
            SELECT
                o.omi_id_operacion,
                o.c063,
                REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*') AS ce_token,
                CASE
                    WHEN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*') IS NOT NULL
                        AND POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) > 0
                    THEN SUBSTRING(
                        REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
                        FROM POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
                        FOR 2
                    )
                    ELSE NULL
                END AS leading_indicator,
                CASE
                    WHEN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*') IS NULL
                        THEN 'NO_3DS'
                    WHEN SUBSTRING(
                            REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
                            FROM POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
                            FOR 2
                        ) IN ('kA','kB','kC','kE','kF','kJ','kR','kS','kG','kO','kP')
                        THEN '3DS_AUTHENTICATED'
                    WHEN SUBSTRING(
                            REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
                            FROM POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
                            FOR 2
                        ) IN ('kN','kW','kU','kX')
                        THEN '3DS_NOT_AUTHENTICATED'
                    ELSE 'UNKNOWN'
                END AS three_ds_status,
                CASE
                    WHEN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*') IS NULL
                        THEN 'NO_3DS'
                    WHEN SUBSTRING(
                            REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
                            FROM POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
                            FOR 2
                        ) IN ('kA','kC','kE','kO')
                        THEN 'FRICTIONLESS'
                    WHEN SUBSTRING(
                            REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
                            FROM POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
                            FOR 2
                        ) IN ('kB','kS','kG','kP')
                        THEN 'CHALLENGE'
                    WHEN SUBSTRING(
                            REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')
                            FROM POSITION('01' IN REGEXP_SUBSTR(o.c063, 'CE[0-9]{5}[^!]*')) + 2
                            FOR 2
                        ) IN ('kN','kW','kU','kX')
                        THEN 'EXEMPT_OR_INFO'
                    ELSE 'UNKNOWN'
                END AS three_ds_flow
            FROM oimt_base o
        )

        SELECT
            tc.*,
            pv.metodo_verificacion AS metodo_identificacion,
            td.leading_indicator,
            td.three_ds_status,
            td.three_ds_flow,
            td.c063,
            o.sucursal AS afiliacion,
            o.terminal AS numero_terminal,
            ob.adquirente
        FROM tc_base tc
        LEFT JOIN pin_verif pv
            ON pv.omi_id_operacion = tc.omi_id_operacion_raw
        LEFT JOIN three_ds td
            ON td.omi_id_operacion = tc.omi_id_operacion_raw
        LEFT JOIN is_pii_parabilium.parabilium_transactions pt
            ON pt.id = tc.omi_id_operacion_raw
        LEFT JOIN is_pii_parabilium.operaciones o
            ON o.id_operacion = tc.omi_id_operacion_raw
        LEFT JOIN oimt_base ob
            ON ob.omi_id_operacion = tc.omi_id_operacion_raw
    """

    cb_df = pd.read_sql(cb_query, get_db_conn())
    if cb_df.empty:
        return cb_df

    cb_df["trx_time"] = cb_df["trx_timestamp_mx"]
    cb_df["trx_timestamp_mx"] = pd.to_datetime(cb_df["trx_timestamp_mx"])
    cb_df["cb_timestamp"] = pd.to_datetime(cb_df["cb_timestamp"])
    cb_df["week_number"] = cb_df["trx_timestamp_mx"].dt.isocalendar().week
    cb_df["trx_date"] = cb_df["trx_timestamp_mx"].dt.date
    cb_df["trx_month"] = cb_df["trx_timestamp_mx"].dt.month
    cb_df["trx_day"] = cb_df["trx_timestamp_mx"].dt.day
    cb_df["trx_month_year"] = cb_df["trx_timestamp_mx"].dt.to_period("M")
    cb_df["trx_week_year"] = cb_df["trx_timestamp_mx"].dt.strftime("%Y-%W")
    cb_df["cb_date"] = cb_df["cb_timestamp"].dt.date
    cb_df["cb_month"] = cb_df["cb_timestamp"].dt.month
    cb_df["cb_day"] = cb_df["cb_timestamp"].dt.day
    cb_df["cb_month_year"] = cb_df["cb_timestamp"].dt.to_period("M")
    cb_df["cb_week_year"] = cb_df["cb_timestamp"].dt.strftime("%Y-%W")
    cb_df.drop_duplicates(subset=["transaction_id"], inplace=True, keep="first")
    cb_df = cb_df[(cb_df["transaction_id"].notna()) & (cb_df["amount"].notna())]
    cb_df = cb_df[cb_df["cb_reason"] != "Devolución de dinero"]
    cb_df["cb_reason_original"] = cb_df["cb_reason"]
    cb_df["cb_reason"] = cb_df["cb_reason"].map(
        {
            "Transacción por internet o compra telefónica": "Rembolso no recibido",
            "Error durante el proceso": "Cargo no reconocido",
            "Otro": "Otro",
            "Producto no recibido": "Rembolso no recibido",
            "Transacción duplicada": "Transacción duplicada",
            "Transacción declinada": "Transacción declinada",
            "Transacción excede el importe autorizado": "Rembolso no procesado",
            "Aclaración de Transacción de Cajero Automático": "Aclaración de Transacción de Cajero Automático",
            "ATM no entrega dinero": "Aclaración de Transacción de Cajero Automático",
            "Reporte de cargo no reconocido": "Cargo no reconocido",
            "TRANSACTION_EXCEEDS_AUTHORIZED_AMOUNT": "Rembolso no procesado",
            "Reembolso no procesado": "Rembolso no procesado",
            "REFUND_NOT_PROCESSED": "Rembolso no procesado",
            "TRANSACTION_DECLINED": "Transacción declinada",
            "OTHER": "Otro",
            "UNRECOGNIZED_TRANSACTION": "Cargo no reconocido",
            "INTERNET_PHONE_TRANSACTION": "Rembolso no recibido",
            "UNRECOGNIZED_CHARGE": "Cargo no reconocido",
            "DUPLICATE_TRANSACTION": "Transacción duplicada",
            "ATM_TRANSACTION_CLARIFICATION": "Aclaración de Transacción de Cajero Automático",
        }
    )
    cb_df["country"] = cb_df["merchant"].str[-2:]
    return cb_df


def chargeback_merchant_summary(
    merchant_name: str,
    blocking_date: str | dt.date | dt.datetime | pd.Timestamp,
) -> Dict[str, Dict]:
    report: Dict[str, Dict] = {}
    merchant_name = merchant_name.replace("*", "")
    blocking_ts = _normalize_blocking_timestamp(blocking_date)
    timeframes = _build_timeframes(blocking_date)
    merchant_cb_df = find_chargeback(merchant_name, blocking_date)

    if merchant_cb_df.empty:
        report[merchant_name] = {
            "merchant_name": merchant_name,
            "blocking_date": str(blocking_ts),
            "first_appearance": None,
            "last_appearance": None,
            "shared_commerces_summary": [],
            "shared_commerces_transactions": [],
            "timeframes": {},
        }
        return report

    merchant_summary = {
        "merchant_name": merchant_name,
        "blocking_date": str(blocking_ts),
        "first_appearance": str(merchant_cb_df["trx_timestamp_mx"].min()),
        "last_appearance": str(merchant_cb_df["trx_timestamp_mx"].max()),
        "timeframes": {},
    }

    for timeframe_name, bounds in timeframes.items():
        start_time = bounds["start_time"]
        end_time = bounds["end_time"]
        timeframe_df = merchant_cb_df[
            (merchant_cb_df["trx_timestamp_mx"] >= start_time)
            & (merchant_cb_df["trx_timestamp_mx"] <= end_time)
        ].copy()

        merchant_summary["timeframes"][timeframe_name] = {
            "start_time": str(start_time),
            "end_time": str(end_time),
            "num_chargebacks_reported": int(timeframe_df["transaction_id"].nunique()),
            "num_unique_users": int(timeframe_df["user_id"].nunique()),
            "num_klrids_per_card_type": _series_to_dict(
                timeframe_df.groupby("card_type")["klrid"].nunique()
            ),
            "num_klrids_per_product_type": _series_to_dict(
                timeframe_df.groupby("product_type")["klrid"].nunique()
            ),
            "num_transactions_per_cvv_indicator": _series_to_dict(
                timeframe_df.groupby("cvv_ind")["transaction_id"].nunique()
            ),
            "num_transactions_per_3ds_status": _series_to_dict(
                timeframe_df.groupby("three_ds_status")["transaction_id"].nunique()
            ),
        }

    report[merchant_name] = merchant_summary
    return report


In [18]:
def build_cod_risk_summary(merchant_to_block: str, block_date: str) -> str:

    ''' 
    INCLUDE OPERADOR NOT MERCHANT NAME 
    '''
    response_code_query = f"""
        WITH daily_codes AS (
        SELECT
            date_trunc('day', fecharegistro) AS day,
            operador,
            cod_respuesta,
            count(*) AS transaction_count,
            sum(importe) AS total_importe
        FROM is_pii_parabilium.operations_temp ot
        WHERE operador ILIKE '%' || '{merchant_to_block}' || '%'
            AND fecharegistro >= DATE '2025-01-05'
            and fecharegistro <= DATE '{block_date}'
        GROUP BY 1, 2, 3
        ),
        daily_totals AS (
        SELECT
            day,
            operador,
            sum(transaction_count) AS total_txn_day,
            sum(total_importe) AS total_importe_day
        FROM daily_codes
        GROUP BY 1, 2
        ),
        risky_codes AS (
        SELECT
            dc.day,
            dc.operador,
            dc.cod_respuesta,
            dc.transaction_count,
            dc.total_importe,
            dt.total_txn_day,
            ROUND(100.0 * dc.transaction_count / NULLIF(dt.total_txn_day, 0), 2) AS pct_of_day_txns,
            LAG(dc.transaction_count) OVER (
                PARTITION BY dc.operador, dc.cod_respuesta
                ORDER BY dc.day
            ) AS prev_txn_count,
            LAG(dc.total_importe) OVER (
                PARTITION BY dc.operador, dc.cod_respuesta
                ORDER BY dc.day
            ) AS prev_importe
        FROM daily_codes dc
        JOIN daily_totals dt
            ON dc.day = dt.day
        AND dc.operador = dt.operador
        WHERE dc.cod_respuesta IN ('87', '14')
        )
        SELECT
            day,
            operador,
            cod_respuesta,
            transaction_count,
            total_importe,
            total_txn_day,
            pct_of_day_txns,
            prev_txn_count,
            CASE
            WHEN prev_txn_count IS NULL OR prev_txn_count = 0 THEN NULL
            ELSE ROUND(100.0 * (transaction_count - prev_txn_count) / prev_txn_count, 2)
            END AS pct_change_vs_prev_day,
            CASE
            WHEN prev_txn_count IS NULL THEN 'no prior data'
            WHEN transaction_count > prev_txn_count THEN 'increasing'
            WHEN transaction_count < prev_txn_count THEN 'decreasing'
            ELSE 'flat'
            END AS trend_direction
        FROM risky_codes
        ORDER BY day DESC, operador, cod_respuesta;
    """
    df = pd.read_sql_query(response_code_query, con=get_db_conn())
    
    # Ensure sorting (latest first)
    df = df.sort_values(["day", "cod_respuesta"], ascending=[False, True])
    
    summaries = []
    
    for day, group in df.groupby("day", sort=False):
        operador = group["operador"].iloc[0].strip()
        total_txn = int(group["total_txn_day"].iloc[0])
        
        day_summary = [f"Date: {day.date()} | Merchant: {operador} | Total transactions: {total_txn}."]
        
        for _, row in group.iterrows():
            code = row["cod_respuesta"]
            txns = int(row["transaction_count"])
            pct = row["pct_of_day_txns"]
            trend = row["trend_direction"]
            change = row["pct_change_vs_prev_day"]
            
            # Handle NaNs nicely
            if pd.isna(change):
                change_text = "no prior comparison"
            else:
                change_text = f"{change:.2f}% vs previous day"
            
            line = (
                f"Code {code}: {txns} transactions "
                f"({pct:.2f}% of volume), trend is {trend}"
                f" ({change_text})."
            )
            
            day_summary.append(line)
        
        summaries.append(" ".join(day_summary))
    
    return "\n".join(summaries)

In [19]:
from typing import Any, Dict

def _normalize_prompt_value(value: Any) -> str:
    if value is None:
        return "null"

    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return "null"

    return text


def format_commerce_summary_for_prompt(commerce_summary: pl.DataFrame | Any) -> str:
    """
    Convert the Polars merchant summary dataframe into a compact, readable
    prompt section instead of dumping the dataframe object with json.dumps.
    """
    if commerce_summary is None:
        return "No transactional summary available."

    if not isinstance(commerce_summary, pl.DataFrame):
        return json.dumps(commerce_summary, indent=2, default=str)

    if commerce_summary.is_empty():
        return "No transactional summary available."

    expected_columns = {"metric", "t1d", "t1w", "ytd"}
    if expected_columns.issubset(set(commerce_summary.columns)):
        lines = []
        for row in commerce_summary.iter_rows(named=True):
            metric = row.get("metric", "unknown_metric")
            t1d = _normalize_prompt_value(row.get("t1d"))
            t1w = _normalize_prompt_value(row.get("t1w"))
            ytd = _normalize_prompt_value(row.get("ytd"))
            lines.append(f"- {metric}: t1d={t1d}, t1w={t1w}, ytd={ytd}")
        return "\n".join(lines)

    return commerce_summary.write_json(row_oriented=True)


In [20]:
def build_card_verification_query(merchant: str) -> str:
    merchant = merchant.replace("'", "''")  # escape single quotes for SQL

    return f"""
WITH params AS (
    SELECT
        date_trunc('hour', current_timestamp) - interval '1 hour' AS current_hour_start,
        date_trunc('hour', current_timestamp) AS current_hour_end
),
base AS (
    SELECT
        ot.operador,
        RIGHT(ot.operador, 2) AS pais,
        ot.fecharegistro,
        ot.importe
    FROM is_pii_parabilium.operations_temp ot
    LEFT JOIN is_provider_parabilium.auth_blacklist_elements bl
        ON LEFT(ot.operador, 22) = LEFT(bl.value, 22)
    CROSS JOIN params p
    WHERE
        ot.operador = '{merchant}'
        AND RIGHT(ot.operador, 2) NOT IN ('MX', 'Mx')
        AND bl.value IS NULL
        AND ot.fecharegistro >= p.current_hour_start - interval '7 days'
        AND ot.fecharegistro < p.current_hour_end
),
current_hour AS (
    SELECT
        operador,
        pais,
        COUNT(*) AS total_count_1h,
        SUM(CASE WHEN importe = 0 THEN 1 ELSE 0 END) AS validation_count_1h,
        SUM(CASE WHEN importe = 0 THEN 1 ELSE 0 END)::float / NULLIF(COUNT(*), 0) AS validation_rate_1h
    FROM base b
    CROSS JOIN params p
    WHERE
        b.fecharegistro >= p.current_hour_start
        AND b.fecharegistro < p.current_hour_end
    GROUP BY
        operador,
        pais
),
historical_same_hour AS (
    SELECT
        operador,
        date_trunc('hour', fecharegistro) AS hora,
        COUNT(*) AS total_count_hour,
        SUM(CASE WHEN importe = 0 THEN 1 ELSE 0 END) AS validation_count_hour,
        SUM(CASE WHEN importe = 0 THEN 1 ELSE 0 END)::float / NULLIF(COUNT(*), 0) AS validation_rate_hour
    FROM base b
    CROSS JOIN params p
    WHERE
        b.fecharegistro >= p.current_hour_start - interval '7 days'
        AND b.fecharegistro < p.current_hour_start
        AND EXTRACT(hour FROM b.fecharegistro) = EXTRACT(hour FROM p.current_hour_start)
    GROUP BY
        operador,
        date_trunc('hour', fecharegistro)
),
historical_baseline AS (
    SELECT
        operador,
        AVG(total_count_hour) AS avg_total_count_same_hour_7d,
        AVG(validation_count_hour) AS avg_validation_count_same_hour_7d,
        AVG(validation_rate_hour) AS avg_validation_rate_same_hour_7d,
        MAX(validation_count_hour) AS max_validation_count_same_hour_7d,
        MAX(validation_rate_hour) AS max_validation_rate_same_hour_7d,
        COUNT(*) AS baseline_hours
    FROM historical_same_hour
    GROUP BY
        operador
)
SELECT
    c.operador,
    c.pais,
    c.total_count_1h,
    c.validation_count_1h,
    c.validation_rate_1h,
    h.avg_total_count_same_hour_7d,
    h.avg_validation_count_same_hour_7d,
    h.avg_validation_rate_same_hour_7d,
    h.max_validation_count_same_hour_7d,
    h.max_validation_rate_same_hour_7d,
    h.baseline_hours,
    c.validation_count_1h - h.avg_validation_count_same_hour_7d AS validation_count_lift,
    c.validation_rate_1h / NULLIF(h.avg_validation_rate_same_hour_7d, 0) AS validation_rate_multiplier
FROM current_hour c
LEFT JOIN historical_baseline h
    ON c.operador = h.operador
WHERE
    c.validation_count_1h >= 5 ------ MODIFY
    AND c.validation_rate_1h >= 0.40
ORDER BY
    c.validation_count_1h DESC;
"""

def summarize_card_verification_anomaly(merchant: str) -> str:

    query = build_card_verification_query(merchant)
    card_verification_df = pd.read_sql(query, con=get_db_conn())
    if card_verification_df is None or card_verification_df.empty:
        return "No card verification anomaly matched the configured thresholds in the last hour."

    required_columns = [
        "operador",
        "validation_count_1h",
        "validation_rate_1h",
        "avg_validation_count_same_hour_7d",
        "avg_validation_rate_same_hour_7d",
        "validation_rate_multiplier",
    ]
    missing_columns = [col for col in required_columns if col not in card_verification_df.columns]
    if missing_columns:
        raise ValueError(f"Missing expected columns: {', '.join(missing_columns)}")

    def _clean_value(value):
        return None if pd.isna(value) else value

    def _format_number(value, decimals: int = 1) -> str:
        value = _clean_value(value)
        if value is None:
            return "N/A"
        return f"{value:.{decimals}f}"

    def _format_pct(value) -> str:
        value = _clean_value(value)
        if value is None:
            return "N/A"
        return f"{value * 100:.1f}%"

    def _summarize_row(row: pd.Series) -> str:
        merchant = str(_clean_value(row["operador"]) or "Unknown merchant").strip()
        current_count = _clean_value(row["validation_count_1h"])
        current_rate = _clean_value(row["validation_rate_1h"])
        avg_count = _clean_value(row["avg_validation_count_same_hour_7d"])
        avg_rate = _clean_value(row["avg_validation_rate_same_hour_7d"])
        multiplier = _clean_value(row["validation_rate_multiplier"])

        if avg_count is None or avg_rate is None:
            return (
                f"{merchant} had {_format_number(current_count, 0)} card verifications in the last hour, "
                f"with a verification rate of {_format_pct(current_rate)}. "
                "No reliable 7-day same-hour baseline was available, so this should be reviewed manually."
            )

        return (
            f"{merchant} had {_format_number(current_count, 0)} card verifications in the last hour, "
            f"versus a 7-day same-hour average of {_format_number(avg_count)}. "
            f"The current verification rate is {_format_pct(current_rate)}, compared with a historical "
            f"same-hour average of {_format_pct(avg_rate)}. "
            f"That is about {_format_number(multiplier)}x the normal verification rate, suggesting abnormal "
            "card-testing or card-validation behavior."
        )

    summaries = [_summarize_row(row) for _, row in card_verification_df.iterrows()]
    return summaries[0] if len(summaries) == 1 else "\n".join(f"- {summary}" for summary in summaries)


In [21]:
def build_final_report(
    merchant: str,
    blocked_date: str,
    commerce_summary: pl.DataFrame,
    chargebacks_summary: Dict,
    cod_analysis_df: str,
    alert_type: str,
    card_verifications_summary: str = "",
) -> str:
    """
    Standalone version of the notebook prompt builder that formats the Polars
    commerce summary before embedding it into the prompt.
    """
    formatted_commerce_summary = format_commerce_summary_for_prompt(commerce_summary)
    formatted_chargebacks_summary = (
        json.dumps(chargebacks_summary, indent=2, default=str)
        if chargebacks_summary
        else "No chargeback records found for the specified merchant."
    )
    formatted_cod_analysis = (
        cod_analysis_df
        if cod_analysis_df
        else "No response code data available for analysis."
    )
    formatted_card_verifications_summary = str(card_verifications_summary).strip() if card_verifications_summary else ""
    if not formatted_card_verifications_summary or formatted_card_verifications_summary.lower() == "nan":
        formatted_card_verifications_summary = "No card verification anomaly summary available."


    fraud_prompt_final = f"""
    MERCHANT FRAUD ANALYSIS REQUEST for alert {alert_type}
    ================================

    MERCHANT CONTEXT:
    {merchant} was blocked on {blocked_date} due to "Card Verifications".

    ---

    PAST TRANSACTIONALITY SUMMARY:
    {formatted_commerce_summary}

    ---

    CHARGEBACKS SUMMARY:
    {formatted_chargebacks_summary}

    ---

    RESPONSE CODES ANALYSIS:
    {formatted_cod_analysis}

    ---

    CARD VERIFICATIONS SUMMARY:
    {formatted_card_verifications_summary}

    ---

    ANALYSIS REQUIREMENTS:
    1. Evaluate the merchant based on all three data dimensions (transactions, chargebacks, response codes)
    2. Identify patterns that indicate fraud or card testing activities
    3. Consider the relationship between failed transaction counts and specific response codes
    4. Assess the risk of this merchant based on:
    - Transaction velocity and volume
    - Chargeback patterns and frequency
    - Response code distribution (especially codes 14, 87, 51)
    - Temporal patterns in activity
    5. Provide a clear recommendation: BLOCK or DO NOT BLOCK

    Make a comprehensive decision based on all provided data.
    """
    return fraud_prompt_final


In [22]:
def prompt_pipeline(merchant: str, blocked_date: str, alert_type: str) -> str:
    trx_summary_df = merchant_trx_summary(merchant=merchant, blocked_date=blocked_date)
    # create chargeback summary for the merchant
    cb_df = chargeback_merchant_summary(merchant_name=merchant, blocking_date=blocked_date)
    # create cod response df
    cod_df = build_cod_risk_summary(merchant_to_block=merchant, block_date=blocked_date)
    # create card verification df
    cv_summary = summarize_card_verification_anomaly(merchant=merchant)

    # build the final report for the agent
    user_prompt = build_final_report(
        merchant=merchant,
        blocked_date=blocked_date,
        commerce_summary=trx_summary_df,
        chargebacks_summary=cb_df,
        cod_analysis_df=cod_df,
        card_verifications_summary=cv_summary,
        alert_type=alert_type,
    )
    
    return user_prompt

In [23]:
async def fraud_CV_agent(final_prompt: str) :
    # read prompt txt from file
    with open("/Users/camila.cusicanqui/Documents/klar/mini-tasks/agents-automization/fraud_alerts_instruction_prompt.txt", "r") as file:
        instructions_fraud = file.read()
        
    agent_fraude = Agent(
        name="Fraud Analyst",
        instructions=instructions_fraud,
        output_type=FraudFinding,
    )
    result = await Runner.run(
        agent_fraude,
        final_prompt,
    )
    return result
    #print(result)

In [24]:
SERVICE_ACCOUNT_FILE = r'.config/klar-cami-cusi.json'
gc = gspread.service_account(filename=SERVICE_ACCOUNT_FILE)
sheet_id = "15JrCjqynUuPd8kXlixAIIYeYNcADiG48GeSLxeIy0ks"
worksheet_gid = '601362568'
spreadsheet = gc.open_by_key(sheet_id)
# connect to ghseets
worksheet = spreadsheet.worksheet('actual_bl')
bl_merchants_df = pd.DataFrame(worksheet.get_all_records())
bl_merchants_df.rename(columns={'merchant_name': 'operador'}, inplace=True)
bl_merchants_df["updated_at"] = pd.to_datetime(bl_merchants_df["updated_at"])
# remove the * from the merchant names
bl_merchants_df["merchant"] = bl_merchants_df["operador"].str.replace('*', '', regex=False)
bl_merchants_df["operador_length"] = bl_merchants_df["operador"].apply(lambda x: len(x))
bl_merchants_df["merchant_length"] = bl_merchants_df["merchant"].apply(lambda x: len(x))

# obtain the merchants that we need to analyze from the BL list, filter by date and reason of block
bl_merchants_to_test = bl_merchants_df[
    (bl_merchants_df["updated_at"] >= '2026-01-01')
    & (bl_merchants_df["status"] == 'Bloqueado')
    & (bl_merchants_df["reason"] == 'Card Verifications')
][["merchant", 'updated_at']]


In [32]:
len('ANTHROPIC             SAN FRANCISCOCA US')

40

In [25]:
# TODO: also consider the related commerce summary for the shared commerce analysis

In [26]:
bl_merchants_to_test.iloc[-2:]

,merchant,updated_at
359,WWW.TECH-VAULT.COM SINGAPORE SG,2026-02-07 07:05:59
361,OBERWEIS HOME DELIVERYNORTH AURORA IL US,2026-01-31 04:06:34


In [33]:
# antropic_test
TEST_MERCHANT = 'ANTHROPIC             SAN FRANCISCOCA US'
TEST_BLOCKED_DATE = '2026-04-01 05:07:36'
print(f"Analyzing merchant: {TEST_MERCHANT} | Blocked at: {TEST_BLOCKED_DATE}")
print("Fetching transaction data summary...")
# obtain the transaction summary for the merchant
user_prompt = prompt_pipeline(
    merchant=TEST_MERCHANT,
    blocked_date=TEST_BLOCKED_DATE,
    alert_type= 'ABNORMAL CARD VERIFICATIONS'
)
print("Final comprehensive prompt for analysis:")
result = await fraud_CV_agent(user_prompt)
print("Agent result:")
result = result.final_output.model_dump()  # Pydantic v2
result["suspicious_signals"] = "\n".join(result["suspicious_signals"])
print(result)

print('Finished analyzing merchant:', TEST_MERCHANT, '.....\n')
print(' ')
print(' ')

Analyzing merchant: ANTHROPIC             SAN FRANCISCOCA US | Blocked at: 2026-04-01 05:07:36
Fetching transaction data summary...


/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820/2016719537.py:194: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cb_df = pd.read_sql(cb_query, get_db_conn())
/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820/2004360204.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(response_code_query, con=get_db_conn())
/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820/3607359053.py:100: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  card_verification_df = pd.read_s

Final comprehensive prompt for analysis:
Agent result:
{'merchant_name': 'ANTHROPIC             SAN FRANCISCOCA US', 'risk_level': 'high', 'summary': 'The merchant demonstrates persistent abnormal card verification rates (72.7% vs 67.2% baseline), an elevated and increasing rate of risky code 87 response (with spikes up to 46% of daily transactions), and 2 chargebacks YTD (both on cards with no CVV and unknown 3DS). Transaction history shows significant failed/canceled states and a notable percentage of no-CVV/no-3DS attempts. These combined factors, alongside the already-blocked status for card verifications, show a strong and repeated pattern of card-testing or validation abuse.', 'suspicious_signals': 'Abnormal card verification rate currently 72.7% vs 67.2% baseline (1.1x lift)\nFrequent and sometimes high concentration of response code 87 (e.g., >10% many days, spikes up to 46.15% of daily volume)\nTrend of code 87 spikes and instability in use, including recent increases\n2 charg

In [27]:
extra_merchant_result = {}

for merchant, blocked_date in bl_merchants_to_test.drop(index=[3,7,8]).itertuples(index=False):
# bl_merchants_to_test.drop([0,1]).itertuples(index=False):
    blocked_date = str(blocked_date)
    print(f"Analyzing merchant: {merchant} | Blocked at: {blocked_date}")
    print("Fetching transaction data summary...")
    # obtain the transaction summary for the merchant
    user_prompt = prompt_pipeline(
        merchant=merchant,
        blocked_date=blocked_date,
        alert_type= 'ABNORMAL CARD VERIFICATIONS'
    )
    print("Final comprehensive prompt for analysis:")
    result = await fraud_CV_agent(user_prompt)
    print("Agent result:")
    result = result.final_output.model_dump()  # Pydantic v2
    result["suspicious_signals"] = "\n".join(result["suspicious_signals"])
    print(result)
    df = pd.DataFrame([result])
    extra_merchant_result[merchant] = df
    print('Finished analyzing merchant:', merchant, '.....\n')
    print(' ')
    print(' ')

Analyzing merchant: FOUNDERSCARD.COM      NEW YORK     NY US | Blocked at: 2026-05-06 18:48:05
Fetching transaction data summary...


/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820/2016719537.py:194: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cb_df = pd.read_sql(cb_query, get_db_conn())
/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820/2004360204.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(response_code_query, con=get_db_conn())
/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820/3607359053.py:100: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  card_verification_df = pd.read_s

Final comprehensive prompt for analysis:
Agent result:
{'merchant_name': 'FOUNDERSCARD.COM      NEW YORK     NY US', 'risk_level': 'high', 'summary': "The merchant has only 2 transactions on record, both occurring within a short timeframe (t1w: 2026-05-05 09:26-09:27), from a single user. Both transactions returned response code 87 (CVV incorrect/missing), indicating a 100% failure rate due to invalid CVV information—a strong card-testing indicator. There is no normal authorization or settlement activity beyond these, and no card verifications exceeded alert thresholds, possibly due to low attempt volume. No chargebacks are reported yet, but the absence is not exculpatory given the merchant's newness and very limited history.", 'suspicious_signals': 'Only 2 transactions, both within 1 minute window\n100% failed transactions with response code 87 (CVV incorrect/missing)\nTransactions from a single unique user\nNo normal transaction activity\nSudden merchant activation and immediate fail

/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820/2016719537.py:194: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cb_df = pd.read_sql(cb_query, get_db_conn())
[non-fatal] Tracing client error 429: {
  "error": {
    "message": "You've exceeded the rate limit, please slow down and try again after 60.055806 seconds.",
    "type": "invalid_request_error",
    "param": null,
    "code": "rate_limit_exceeded"
  }
}
/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820/2004360204.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(response_code_query, con=get_db_conn())
/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820

Final comprehensive prompt for analysis:
Agent result:
{'merchant_name': 'EMPATHIQ              SAN DIEGO    CA US', 'risk_level': 'high', 'summary': 'The merchant shows extremely recent activity with a high concentration of risky response codes: code 14 (invalid card) and code 87 (CVV errors) together make up a large share of the past days’ transactions (up to 100%), with only 4 unique users and previously little to no history. No chargebacks are present, but the strong pattern of failed and declined card validations, repeated in a short window, and a very low user count, indicate clear card-testing or validation attacks.', 'suspicious_signals': 'Very recent first and last transactions, merchant is likely new\nExtremely low unique user count (2 in t1d, 4 in t1w/ytd)\nCode 14 (invalid card) and 87 (CVV error) account for up to 100% of daily volume\nCode 87 spiking from 14% to 67% of volume day-over-day\nHigh count of failed/canceled transactions relative to total\nNo card verification 

...

Final comprehensive prompt for analysis:
Agent result:
{'merchant_name': 'TRIVIVA LLC           ROUND ROCK   TX US', 'risk_level': 'high', 'summary': 'This merchant exhibited extremely suspicious transactional behavior within a single day of activity, including a high number of failed card verifications (codes 14, 87, and 51), all occurring without CVV or 3DS, and generated 4 unique-user chargebacks in its brief active window. The consolidation of all transactional activity within one day, high failed/canceled rates compared to settled (22 failed vs 7 settled), and every chargeback showing no CVV and PHYSICAL card type, strongly indicate card testing and fraud.', 'suspicious_signals': 'All activity occurred within a single day (t1d = t1w = ytd = 2026-02-07)\nHigh failed/canceled versus authorized/settled ratio (22 failed vs 7 settled)\nAll card verifications (7) lacked 3DS and CVV\nHigh counts of card-testing codes: 14 (19), 87 (26), 51 (22), all within first day\nOnly 29 unique users 

/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820/2016719537.py:194: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cb_df = pd.read_sql(cb_query, get_db_conn())
/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820/2004360204.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(response_code_query, con=get_db_conn())
/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820/3607359053.py:100: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  card_verification_df = pd.read_s

Final comprehensive prompt for analysis:
Agent result:
{'merchant_name': 'P&C SHOP /SHIPPELLO   LAUDERDALE LAFL US', 'risk_level': 'high', 'summary': 'This merchant demonstrates a clear card testing/card validation pattern with very recent and limited transaction history, a high count of response code 14 errors (invalid card numbers), significant failed and non-authenticated attempts, and markedly abnormal distribution of high-risk response codes. The proportion of failed or card-verification transactions to total activity is extremely high, and card activity is concentrated in a short window after a dormant period, all without chargeback evidence (which is common with new merchants).', 'suspicious_signals': 'Very recent surge in activity after little prior history (first tx t1d=2026-02-07, most activity same day)\ncnt_cr_14=23 (invalid card number), huge compared to only 3-5 authorized/settled txs\ncnt_cr_87=2-5, strong card testing indicator\nFailed/canceled transactions (t1d=4) comp

/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_14820/2016719537.py:194: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cb_df = pd.read_sql(cb_query, get_db_conn())


DatabaseError: Execution failed on sql: 
        WITH tc_base AS (
            SELECT
                tc.*,
                COALESCE(
                    NULLIF(split_part(tc.transaction_id, 'PARABILIUM:', 2), ''),
                    tc.transaction_id
                ) AS omi_id_operacion_raw
            FROM ops_fraud.total_chargeback tc
            WHERE
                tc.merchant = 'SHIPPELLO             GARLAND      TX US'
                AND tc.trx_timestamp_mx < DATEADD(day, 1, DATE '2026-02-07')
        ),

        oimt_base AS (
            SELECT
                oimt.omi_id_operacion,
                oimt.c063,
                oimt.c032 AS adquirente
            FROM is_pii_parabilium.operation_iso_messages_temp oimt
            INNER JOIN tc_base tc
                ON oimt.omi_id_operacion = tc.omi_id_operacion_raw
        ),

        pin_verif AS (
            SELECT
                o.omi_id_operacion,
                '! ' || REGEXP_SUBSTR(o.c063, 'B300080[^!]*') AS B30008

In [181]:
extra_result_df = pd.concat(
    [
        df.assign(merchant_name=merchant_name)
        for merchant_name, df in extra_merchant_result.items()
    ],
    ignore_index=True
)

In [180]:
extra_merchant_result["WWW.TECH-VAULT.COM    SINGAPORE       SG"]

,risk_level,summary,suspicious_signals,recommendation,confidence
0,high,Merchant WWW.TECH-VAULT.COM demonstrates clear...,High count of response code 87 (fraud/security...,BLOCK,0.95


In [88]:
blocked_df = pd.DataFrame(merchant_result.keys(), columns=["merchant"])

In [90]:
blocked_df["is_blocked"] = 0 

In [109]:
blocked_df.iloc[-10, blocked_df.columns.get_loc("is_blocked")] = 1

In [110]:
blocked_df

,merchant,is_blocked
0,Smoke Club FL LLC Orlando FL US,1
1,Paul Stuart Inc 2126820320 NY US,1
2,M800 LIMITED KOWLOON BAY KowHK,1
3,NY POST DIGITAL MEMBERNEW YORK NY US,1
4,COOK MEMORIAL PUBLIC LIBERTYVILLE IL US,1
5,LUAN MEIRELLES DIAS PARIBEIRAO PRETSP BR,0
6,IONOS INC CHESTERBROOK PA US,1
7,Hims and Hers Inc. San FranciscoCA US,1
8,ELKWEDY PERFUMES AND GDubai 000AE,1
9,ELKWEDY FASHION DESIGNDubai 000AE,1


In [63]:
# testing pipeline

merchant = bl_merchants_to_test["merchant"].iloc[5]
blocked_date = str(bl_merchants_to_test["updated_at"].iloc[0])
# example test run for one merchant
print(f"Analyzing merchant: {merchant} | Blocked at: {blocked_date}")
print("Fetching transaction data summary...") 


Analyzing merchant: NY POST DIGITAL MEMBERNEW YORK     NY US | Blocked at: 2026-04-23 16:09:02
Fetching transaction data summary...


In [112]:
# merchant summary
trx_summary_df = merchant_trx_summary(merchant=merchant, blocked_date=blocked_date)

In [113]:
cb_df = chargeback_merchant_summary(merchant_name=merchant, blocking_date=blocked_date)


/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_51320/2836818482.py:197: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cb_df = pd.read_sql(cb_query, get_db_conn())


In [114]:
cod_df = build_cod_risk_summary(merchant_to_block=merchant, block_date=blocked_date)


/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_51320/2004360204.py:74: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(response_code_query, con=get_db_conn())


In [115]:
cod_df

''

In [116]:
user_prompt = build_final_report(
    merchant=merchant,
    blocked_date=blocked_date,
    commerce_summary=trx_summary_df,
    chargebacks_summary=cb_df,
    cod_analysis_df=cod_df
)

In [119]:
result = await fraud_CV_agent(user_prompt)

RunResult:
- Last agent: Agent(name="Fraud Analyst", ...)
- Final output (FraudFinding):
    {
      "risk_level": "high",
      "summary": "The merchant ELKWEDY FASHION DESIGNDubai 000AE shows strong indications of abuse and possible card testing: extremely short and intense transaction window (first to last TX is just a few hours), very high proportion of failed/canceled transactions (50 vs just 14 settled), heavy concentration of activity among a small number of unique users, high numbers of chargebacks (12 from 1 user, all 'no CVV' and 'UNKNOWN' 3DS), and a notable number of code 14 'invalid card number' responses in a very brief period. These signals match known fraud/card testing behaviors.",
      "suspicious_signals": [
        "12 chargebacks from a single user in one week/month (all same card type and BIN, no CVV, UNKNOWN 3DS)",
        "First and last transaction within a few hours; new merchant with all activity compressed into one brief period",
        "High number of fai

In [ ]:
result

In [117]:
print(user_prompt)


    MERCHANT FRAUD ANALYSIS REQUEST

    MERCHANT CONTEXT:
    ELKWEDY FASHION DESIGNDubai        000AE was blocked on 2026-04-01 05:07:36 due to "Card Verifications".

    ---

    PAST TRANSACTIONALITY SUMMARY:
    - first_tx_date: t1d=2026-04-01 00:00, t1w=2026-03-31 22:46, ytd=2026-03-31 22:46
- last_tx_date: t1d=2026-04-01 04:35, t1w=2026-04-01 04:35, ytd=2026-04-01 04:35
- cnt_cr_0: t1d=0, t1w=14, ytd=14
- cnt_cr_0_no_3ds_no_cvv: t1d=0, t1w=14, ytd=14
- cnt_cr_14: t1d=2, t1w=9, ytd=9
- cnt_cr_51: t1d=0, t1w=41, ytd=41
- cnt_cr_51_no_3ds_no_cvv: t1d=0, t1w=41, ytd=41
- cnt_unique_users: t1d=2, t1w=25, ytd=25
- cnt_state_failed_canceled: t1d=2, t1w=50, ytd=50
- cnt_authorized_settled: t1d=0, t1w=14, ytd=14
- cnt_card_type_PHYSICAL_no_3ds_no_cvv: t1d=0, t1w=38, ytd=38
- cnt_card_type_VIRTUAL_no_3ds_no_cvv: t1d=0, t1w=17, ytd=17
- cnt_product_type_CREDIT_5401_BIN_no_3ds_no_cvv: t1d=0, t1w=51, ytd=51
- cnt_product_type_CREDIT_5456_BIN_no_3ds_no_cvv: t1d=0, t1w=4, ytd=4

    ---

    

In [82]:
for i in merchant_result.keys():
    print(f"Merchant: {i} | Analysis Result: {merchant_result}")


Merchant: Smoke Club FL LLC     Orlando      FL US | Analysis Result: {'Smoke Club FL LLC     Orlando      FL US': None, 'Paul Stuart Inc       2126820320   NY US': None, 'M800 LIMITED          KOWLOON BAY  KowHK': None, 'NY POST DIGITAL MEMBERNEW YORK     NY US': None, 'COOK MEMORIAL PUBLIC  LIBERTYVILLE IL US': None, 'LUAN MEIRELLES DIAS PARIBEIRAO PRETSP BR': None, 'IONOS INC             CHESTERBROOK PA US': None, 'Hims and Hers Inc.    San FranciscoCA US': None, 'ELKWEDY PERFUMES AND GDubai        000AE': None, 'ELKWEDY FASHION DESIGNDubai        000AE': None}
Merchant: Paul Stuart Inc       2126820320   NY US | Analysis Result: {'Smoke Club FL LLC     Orlando      FL US': None, 'Paul Stuart Inc       2126820320   NY US': None, 'M800 LIMITED          KOWLOON BAY  KowHK': None, 'NY POST DIGITAL MEMBERNEW YORK     NY US': None, 'COOK MEMORIAL PUBLIC  LIBERTYVILLE IL US': None, 'LUAN MEIRELLES DIAS PARIBEIRAO PRETSP BR': None, 'IONOS INC             CHESTERBROOK PA US': None, 'Hims an

In [ ]:
# TODO: pruebas % accuracy 

In [122]:
result = await fraud_CV_agent(user_prompt)

In [138]:
df["suspicious_signals"].iloc[0]

'Merchant first appeared less than 24 hours before blocking—very short history\nOnly 25 unique users but 50 failed/canceled transactions and 41 response code 51 (insufficient funds) in a week\n12 chargebacks across only 1 unique user and concentrated in 1 BIN/card type—all without CVV/3DS\nCard testing patterns: high failed/canceled transaction rate, high rc_14 (9/week), high rc_51 (41/week), and 14 authorized/settled across 25 users\nActivity abruptly starts and intensifies, with no prior history'

In [158]:
merchant_result

{'Smoke Club FL LLC     Orlando      FL US':   risk_level                                            summary  \
 0       high  The merchant shows clear evidence of card test...   
 
                                   suspicious_signals recommendation  \
 0  44.83% of transactions were invalid card (code...          BLOCK   
 
    confidence  
 0        0.99  ,
 'Paul Stuart Inc       2126820320   NY US':   risk_level                                            summary  \
 0       high  Paul Stuart Inc exhibited all transaction acti...   
 
                                   suspicious_signals recommendation  \
 0  62 response code 14 (invalid card number) even...          BLOCK   
 
    confidence  
 0        0.98  ,
 'M800 LIMITED          KOWLOON BAY  KowHK':   risk_level                                            summary  \
 0       high  The merchant 'M800 LIMITED KOWLOON BAY KowHK' ...   
 
                                   suspicious_signals recommendation  \
 0  Recent merchant 

In [159]:
merchant_result["AARON ABKE 4D UNIVERS+14082066220 TX US"]

,risk_level,summary,suspicious_signals,recommendation,confidence
0,low,"There is no transactional, chargeback, or resp...",Merchant was blocked for 'Card Verifications.'...,NOT BLOCK,0.6


In [164]:
result_df = pd.concat(
    [
        df.assign(merchant_name=merchant_name)
        for merchant_name, df in merchant_result.items()
    ],
    ignore_index=True
)
result_df = result_df[[
    'merchant_name', 'risk_level', 'summary', 'suspicious_signals', 'recommendation', 'confidence'
]]

In [166]:
result_df.recommendation.value_counts(normalize=True)

recommendation
BLOCK        0.923077
NOT BLOCK    0.076923
Name: proportion, dtype: float64

In [167]:
result_df.shape

(39, 6)

In [169]:
result_df.to_excel('misc_info/fraud_agent_results.xlsx', index=False)

In [171]:
blocking_agent_sheet = '1A7ECmSCOobNxamBdmMzqJqnchO_WjcA68yj-qCrtXcI'

In [172]:
worksheet_blocking_agent = gc.open_by_key(blocking_agent_sheet).worksheet('results')

In [174]:
worksheet_blocking_agent.update([result_df.columns.values.tolist()] + result_df.values.tolist())

{'spreadsheetId': '1A7ECmSCOobNxamBdmMzqJqnchO_WjcA68yj-qCrtXcI',
 'updatedRange': 'results!A1:F40',
 'updatedRows': 40,
 'updatedColumns': 6,
 'updatedCells': 240}

In [170]:
result_df

,merchant_name,risk_level,summary,suspicious_signals,recommendation,confidence
0,Smoke Club FL LLC Orlando FL US,high,The merchant shows clear evidence of card test...,44.83% of transactions were invalid card (code...,BLOCK,0.99
1,Paul Stuart Inc 2126820320 NY US,high,Paul Stuart Inc exhibited all transaction acti...,62 response code 14 (invalid card number) even...,BLOCK,0.98
2,M800 LIMITED KOWLOON BAY KowHK,high,The merchant 'M800 LIMITED KOWLOON BAY KowHK' ...,Recent merchant activity started very recently...,BLOCK,0.99
3,NY POST DIGITAL MEMBERNEW YORK NY US,high,NY POST DIGITAL MEMBERNEW YORK NY US demonstra...,Merchant first observed and only active within...,BLOCK,0.99
4,COOK MEMORIAL PUBLIC LIBERTYVILLE IL US,high,The merchant displayed extremely short and int...,All activity occurred within a single 2-minute...,BLOCK,0.98
5,LUAN MEIRELLES DIAS PARIBEIRAO PRETSP BR,medium,LUAN MEIRELLES DIAS PARIBEIRAO PRETSP BR is a ...,Merchant appeared for the first time with a si...,BLOCK,0.90
6,IONOS INC CHESTERBROOK PA US,high,IONOS INC shows multiple risk signals aligning...,Very short merchant history with significant r...,BLOCK,0.95
7,Hims and Hers Inc. San FranciscoCA US,high,Merchant demonstrates classic card testing and...,100% of 2026-04-03 transactions (25/25) were d...,BLOCK,1.00
8,ELKWEDY PERFUMES AND GDubai 000AE,high,The merchant 'ELKWEDY PERFUMES AND GDubai 000A...,Very recent first and last transaction (same d...,BLOCK,0.98
9,ELKWEDY FASHION DESIGNDubai 000AE,high,The merchant ELKWEDY FASHION DESIGNDubai 000AE...,All activity occurred in less than 2 days (mer...,BLOCK,0.98


In [184]:
# add extra result df to the google sheets worksheet
full_df = pd.concat([result_df, extra_result_df], ignore_index=True)


In [185]:
worksheet_blocking_agent.update([full_df.columns.values.tolist()] + full_df.values.tolist())

{'spreadsheetId': '1A7ECmSCOobNxamBdmMzqJqnchO_WjcA68yj-qCrtXcI',
 'updatedRange': 'results!A1:F42',
 'updatedRows': 42,
 'updatedColumns': 6,
 'updatedCells': 252}

In [ ]:
#create card verification query so that the agent can fetch the data 


In [51]:
merchant = 'ANTHROPIC             SAN FRANCISCOCA US'

In [52]:
test_df = summarize_card_verification_anomaly(merchant)

/var/folders/ch/hb63cwpx22l_1jl5cqj7wp_80000gq/T/ipykernel_9221/3607359053.py:100: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  card_verification_df = pd.read_sql(query, con=get_db_conn())


In [ ]:
test_df

Index(['operador', 'pais', 'total_count_1h', 'validation_count_1h',
       'validation_rate_1h', 'avg_total_count_same_hour_7d',
       'avg_validation_count_same_hour_7d', 'avg_validation_rate_same_hour_7d',
       'max_validation_count_same_hour_7d', 'max_validation_rate_same_hour_7d',
       'baseline_hours', 'validation_count_lift',
       'validation_rate_multiplier'],
      dtype='object')